# v17 · Layered calendrical index: the VAPMIP baseline benchmark, run unmodified
`VAPMIP/benchmarks/hyperwebster_baseline_bench.py` (git 2026-09-25). It loads the canonical `hyperwebster.py` by path.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
bb = load("VAPMIP/benchmarks/hyperwebster_baseline_bench.py", "bb")
HW = bb._load_canonical_hyperwebster()
rows = bb.bench_chunk_sizes(HW, sizes=(140, 500, 1000, 2000, 4000))
print("chars  N_full bits_full  enc_ms   dec_ms | N_min bits_min")
for r in rows: print("%5d  %6d %9d %8.2f %8.2f | %5d %8d" % (r["chars"], r["N_full"], r["addr_bits_full"], r["enc_ms_full"], r["dec_ms_full"], r["N_min"], r["addr_bits_min"]))
ml = bb.bench_multilayer_reconstruction(HW)
print("4-layer chunk/day/month/year reconstruction: %.3f ms, exact: %s" % (ml["total_ms"], ml["roundtrip_ok"]))
assert ml["roundtrip_ok"]

chars  N_full bits_full  enc_ms   dec_ms | N_min bits_min
  140      97       924     0.37     0.30 |    25      649
  500      97      3300     4.29     1.81 |    33     2521
 1000      97      6600    18.38     5.81 |    33     5043
 2000      97     13200    61.55     8.94 |    33    10087
 4000      97     26400   277.41    34.43 |    33    20176
4-layer chunk/day/month/year reconstruction: 42.937 ms, exact: True


The benchmark's sample is one paragraph *repeated* to reach each size, so the 'minimal charset' columns describe a text with ~35 distinct symbols at every size. The reconstruction time is for a chain of four hops through one entry per level: it does not depend on how much history exists. The next notebook measures what the *size* of the address does across layers.